# Train/test split and scaling

one stratified 80/20 split shared by every model and feature set and a scaler fitted (on the training rows only)

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.config import DATA_PATH
from src.imbalance import create_imbalance
from src.features import encode_features, select_feature_sets
from src.split import split_indices, scale

target = 'Performance_Score'
df = pd.read_csv(DATA_PATH)
df_imb = create_imbalance(df, target)

X = encode_features(df_imb, target)
y = df_imb[target]
sets = select_feature_sets(X)

## Stratified split

`stratify=y` keeps the same class proportions in train and test. Without it, class 1 (0.6 % of the rows) could end up almost absent from one side. The split is done once, on row positions, so that the `salary` and `no_salary` sets are evaluated on exactly the same rows.

In [2]:
idx_train, idx_test = split_indices(y)
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

print('train :', len(idx_train), ' test :', len(idx_test))
assert len(idx_train) + len(idx_test) == len(y)
assert len(idx_test) == int(np.ceil(0.2 * len(y)))
assert not set(idx_train) & set(idx_test)

train : 51416  test : 12855


In [3]:
summary = pd.DataFrame({
    'train': y_train.value_counts().sort_index(),
    'test': y_test.value_counts().sort_index(),
})
summary['train_share'] = (summary['train'] / summary['train'].sum()).round(4)
summary['test_share'] = (summary['test'] / summary['test'].sum()).round(4)
summary

,train,test,train_share,test_share
Performance_Score,,,,
1,322,80,0.0063,0.0062
2,3201,801,0.0623,0.0623
3,15999,4000,0.3112,0.3112
4,15952,3988,0.3103,0.3102
5,15942,3986,0.3101,0.3101


In [4]:
assert (summary['train_share'] - summary['test_share']).abs().max() < 0.001


## Scaling

The scaler learns the mean and standard deviation on the **training** rows then applies them to both sides.
Fitting it on all rows would let the test set influence the training (leak). Scaling matters for logistic regression and softmax not for tree models

In [5]:
data = {}
for name, Xs in sets.items():
    X_train, X_test = Xs.iloc[idx_train], Xs.iloc[idx_test]
    X_train_s, X_test_s, scaler = scale(X_train, X_test)
    data[name] = (X_train, X_test, X_train_s, X_test_s)
    print(name, X_train.shape, X_test.shape)

assert data['salary'][0].index.equals(data['no_salary'][0].index)

salary (51416, 26) (12855, 26)
no_salary (51416, 25) (12855, 25)


In [6]:
X_train_s, X_test_s = data['salary'][2], data['salary'][3]

assert np.allclose(X_train_s.mean(), 0, atol=1e-8)
assert np.allclose(X_train_s.std(ddof=0), 1, atol=1e-8)

# the test set is NOT exactly centered: its statistics come from the train set
print('max |test mean| :', X_test_s.mean().abs().max().round(4))
print('test std range  :', X_test_s.std(ddof=0).min().round(3), X_test_s.std(ddof=0).max().round(3))

max |test mean| : 0.0369
test std range  : 0.983 1.022


## Notes

- **Sizes**: 51,416 training rows and 12,855 test rows (80 / 20). Both feature sets (`salary`: 26 columns, `no_salary`: 25) use exactly the same rows, since the split is done once on row positions.
- **Class counts**: train 322 / 3,201 / 15,999 / 15,952 / 15,942 and test 80 / 801 / 4,000 / 3,988 / 3,986 for scores 1 to 5. The shares are identical on both sides (0.63 %, 6.2 %, 31.1 %, 31.0 %, 31.0 %), which is the point of the stratified split.
- **Small test classes**: the test set contains only 80 rows of class 1. One row changes its recall by 1/80 = 0.0125, and the 95 % confidence interval of a recall around 0.5 is about ±0.11. Differences of a few points between models on class 1 are therefore within noise. Class 2 (801 rows) is more stable.
- **Scaling**: the `StandardScaler` is fitted on the training rows only. The scaled training set has mean 0 and standard deviation 1, while the test set does not (largest absolute mean 0.037, standard deviations between 0.983 and 1.022): its statistics come from the training set, which is expected. Fitting the scaler on all rows would leak test information into training.
- **Where scaling matters**: logistic regression and softmax regression. Tree models do not need it.
- **For cross-validation later**: the scaler will have to be refitted inside each fold (with a `Pipeline`), otherwise the validation folds influence the scaling.